# Data prep on Kaggle (CPU): splits + stylegan3 zips from the APTOS competition data

Use this when the lab PC has not produced the dataset yet. Before running:

1. **Add Input → Competitions → APTOS 2019 Blindness Detection** (accept the rules on the competition page first).
2. Optional: add an IDRiD dataset (part B, disease grading) and set `IDRID_INPUT` to its folder; otherwise IDRiD is skipped here and done on the lab PC.
3. Settings: **Internet ON**, CPU is enough. Run with *Save Version → Save & Run All (Commit)*.

Output (`/kaggle/working/out`): `splits/aptos_dedup_dropped.csv`, `splits/aptos_splits.csv`, `sg3/aptos256_f010..f100.zip`, `sg3/aptos256_val.zip`.
Then in the GAN notebooks use **Add Input → Your Work → this notebook** (its output is searched automatically).
Download `splits/*.csv` and commit them to the repo's `splits/` so every machine uses the split the GAN zips were built from.
They hold image ids, split and a grade-blind `frac_rank` only (no labels: APTOS terms, public repo); keep this notebook private.

In [ ]:
REPO = "https://github.com/nabirakhan/genai"
APTOS_INPUT = "auto"      # "auto" finds the folder holding train.csv + train_images under /kaggle/input
IDRID_INPUT = ""          # e.g. "/kaggle/input/idrid-disease-grading" (optional)
MAKE_CACHE = False        # also build the uint8 .npy caches (~800 MB) for the Kaggle classifier fallback

In [ ]:
import os, sys, glob, re, json, time, shutil, signal, subprocess
WORK = "/kaggle/working"; CODE = f"{WORK}/genai"
if not os.path.exists(CODE):
    subprocess.run(["git", "clone", "--recurse-submodules", REPO, CODE], check=True)
else:
    subprocess.run(["git", "-C", CODE, "pull", "--recurse-submodules"], check=False)
os.chdir(CODE)
subprocess.run([sys.executable, "scripts/apply_patches.py"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "click", "requests", "tqdm", "imagehash", "opencv-python-headless", "scikit-learn", "pandas", "scipy", "pillow"], check=True)
for opt in ():    # speed-ups only; stylegan3 falls back without them
    if subprocess.run([sys.executable, "-m", "pip", "install", "-q", opt]).returncode:
        print("optional package not installed:", opt)
if shutil.which("nvidia-smi"):
    print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv"], capture_output=True, text=True).stdout)
import torch; print("torch", torch.__version__, "gpus", torch.cuda.device_count())
def sh(cmd, **kw):
    print("$", " ".join(cmd)); return subprocess.run(cmd, check=True, **kw)


In [ ]:
DATA = f"{WORK}/data"; OUT = f"{WORK}/out"
os.environ["GENAI_DATA"] = DATA
os.environ["GENAI_SPLITS"] = f"{OUT}/splits"
os.makedirs(f"{DATA}/raw/aptos", exist_ok=True); os.makedirs(f"{OUT}/splits", exist_ok=True)
if APTOS_INPUT == "auto":
    found = [os.path.dirname(p) for p in glob.glob("/kaggle/input/**/train.csv", recursive=True)
             if os.path.isdir(os.path.join(os.path.dirname(p), "train_images"))]
    print("APTOS candidates:", found)
    assert found, "add the APTOS 2019 competition as input"
    APTOS_INPUT = found[0]
print("APTOS_INPUT =", APTOS_INPUT)
assert os.path.exists(f"{APTOS_INPUT}/train.csv"), "add the APTOS 2019 competition as input"
if not os.path.exists(f"{DATA}/raw/aptos/train_images"):
    os.symlink(f"{APTOS_INPUT}/train_images", f"{DATA}/raw/aptos/train_images")
shutil.copy(f"{APTOS_INPUT}/train.csv", f"{DATA}/raw/aptos/train.csv")
sh([sys.executable, "-m", "src.data.download_aptos", "--skip-download"])
if IDRID_INPUT:
    os.makedirs(f"{DATA}/raw/idrid", exist_ok=True)
    for name in os.listdir(IDRID_INPUT):
        dst = f"{DATA}/raw/idrid/{name}"
        if not os.path.exists(dst): os.symlink(f"{IDRID_INPUT}/{name}", dst)

In [ ]:
sh([sys.executable, "-m", "src.data.prepare_images", "--source", "both" if IDRID_INPUT else "aptos"])
sh([sys.executable, "-m", "src.data.dedup"])
sh([sys.executable, "-m", "src.data.split"])
sh([sys.executable, "-m", "src.data.make_sg3_zips"])
if MAKE_CACHE:
    sh([sys.executable, "-m", "src.data.cache"])

In [ ]:
shutil.copytree(f"{DATA}/sg3", f"{OUT}/sg3", dirs_exist_ok=True)
if MAKE_CACHE:
    shutil.copytree(f"{DATA}/cache", f"{OUT}/cache", dirs_exist_ok=True)
for root, _, files in os.walk(OUT):
    for f in files:
        p = os.path.join(root, f); print(f"{os.path.getsize(p) / 1e6:8.1f} MB  {p}")
os.chdir(WORK)
shutil.rmtree(DATA, ignore_errors=True)      # keep only /kaggle/working/out in the saved output
shutil.rmtree(CODE, ignore_errors=True)